# REQ_054: Data View Catalog Demo

Demonstrates the `DataViewCatalog` — a universal data view layer that exposes raw analysis data through the same `EpochContext` cursor pattern used by the `ViewCatalog`.

Key pattern: **pin epoch once, access multiple dataviews** without re-specifying the epoch.

```python
ctx = variant.at(epoch=26400)
ctx.dataview("training.metadata.loss_curves").data()     # metadata
ctx.dataview("parameters.embeddings.fourier_coefficients").data()  # per-epoch artifact
ctx.dataview("parameters.pca.trajectory").data()         # cross-epoch artifact
```

## Setup

In [1]:
from miscope import load_family
from miscope.views import dataview_catalog

# Inspect all registered dataviews
print("Registered dataviews:")
for name in dataview_catalog.names():
    print(f"  {name}")

Registered dataviews:
  parameters.embeddings.fourier_coefficients
  parameters.pca.trajectory
  training.metadata.loss_curves


In [2]:
family = load_family("modulo_addition_1layer")
variant = family.get_variant(prime=113, seed=999)
print(f"Loaded variant: {variant.name}")

Loaded variant: modulo_addition_1layer_p113_seed999


## Schema inspection — no IO required

The schema for any dataview is available before loading any data.

In [3]:
from miscope.views.dataview_catalog import _dataview_catalog

for name in dataview_catalog.names():
    dv_def = _dataview_catalog.get(name)
    print(f"\n{name}")
    for field in dv_def.schema.fields:
        print(f"  {field.name} ({field.field_type}): {field.description}")
        print(f"    shape/columns: {field.shape_or_columns}")


parameters.embeddings.fourier_coefficients
  coefficients (ndarray): Fourier coefficients for each embedding dimension at this epoch. Shape: (n_freqs, n_vocab).
    shape/columns: (n_freqs, n_vocab)

parameters.pca.trajectory
  epochs (ndarray): Epoch indices for each row of the PCA projections.
    shape/columns: (n_epochs,)
  projections (ndarray): PCA projections of flattened parameter snapshots (all groups). Shape: (n_epochs, n_components).
    shape/columns: (n_epochs, n_components)
  explained_variance_ratio (ndarray): Fraction of variance explained by each PC (all groups).
    shape/columns: (n_components,)
  explained_variance (ndarray): Absolute variance explained by each PC (all groups).
    shape/columns: (n_components,)
  velocity (ndarray): Parameter update velocity per epoch (all groups). Shape: (n_epochs,).
    shape/columns: (n_epochs,)

training.metadata.loss_curves
  losses (dataframe): Training and test loss at each recorded epoch.
    shape/columns: ['epoch', 'trai

## Shared epoch cursor

Pin an epoch once via `variant.at(epoch)`. All dataviews dispatched through the resulting `EpochContext` are locked to the same training moment.

In [ ]:
# Pin training epoch once
ctx = variant.at(epoch=1000)

# Bind three dataviews to the same epoch — no epoch repeated
loss_dv   = ctx.dataview("training.metadata.loss_curves")
fourier_dv = ctx.dataview("parameters.embeddings.fourier_coefficients")
pca_dv    = ctx.dataview("parameters.pca.trajectory")

print("Bound dataviews:")
print(f"  {loss_dv}")
print(f"  {fourier_dv}")
print(f"  {pca_dv}")

Bound dataviews:
  BoundDataView(name='training.metadata.loss_curves', variant='modulo_addition_1layer_p113_seed999', epoch=26400)
  BoundDataView(name='parameters.embeddings.fourier_coefficients', variant='modulo_addition_1layer_p113_seed999', epoch=26400)
  BoundDataView(name='parameters.pca.trajectory', variant='modulo_addition_1layer_p113_seed999', epoch=26400)


## Data source 1: metadata-based — loss curves

In [5]:
# Access schema before loading
print("Schema (no IO):")
for field in loss_dv.schema.fields:
    print(f"  {field.name}: {field.shape_or_columns}")

# Load data
loss_data = loss_dv.data()
print(f"\nloaded: {loss_data}")
print(f"\nloss_data.losses (first 5 rows):")
loss_data.losses.head()

Schema (no IO):
  losses: ['epoch', 'train_loss', 'test_loss']


ValueError: All arrays must be of the same length

## Data source 2: per-epoch artifact — Fourier coefficients

In [6]:
print("Schema (no IO):")
for field in fourier_dv.schema.fields:
    print(f"  {field.name}: {field.description}")

# Load data
fourier_data = fourier_dv.data()
print(f"\ncoefficients shape: {fourier_data.coefficients.shape}")
print(f"dtype: {fourier_data.coefficients.dtype}")

Schema (no IO):
  coefficients: Fourier coefficients for each embedding dimension at this epoch. Shape: (n_freqs, n_vocab).


FileNotFoundError: No artifact for 'dominant_frequencies' at epoch 26400. Expected: /home/megano/projects/mechinterp/training-dynamics-workbench/results/modulo_addition_1layer/modulo_addition_1layer_p113_seed999/artifacts/dominant_frequencies/epoch_26400.npz

## Data source 3: cross-epoch artifact — PCA trajectory

In [ ]:
print("Schema (no IO):")
for field in pca_dv.schema.fields:
    print(f"  {field.name}: {field.description}")

# Load data
pca_data = pca_dv.data()
print(f"\nepochs shape: {pca_data.epochs.shape}")
print(f"projections shape: {pca_data.projections.shape}")
print(f"explained_variance_ratio: {pca_data.explained_variance_ratio[:3]}")

## Convenience shortcut: variant.dataview()

`variant.dataview(name)` is equivalent to `variant.at(epoch=None).dataview(name)`.  
Per-epoch dataviews resolve to the first available artifact epoch automatically.

In [ ]:
# Shortcut — resolves to first available epoch for per-epoch dataviews
loss_data_shortcut = variant.dataview("training.metadata.loss_curves").data()
print(f"Loss curve via shortcut — {len(loss_data_shortcut.losses)} epochs")
loss_data_shortcut.losses.tail()

## Error handling — unknown dataview name

In [ ]:
try:
    ctx.dataview("not_a_real_dataview")
except KeyError as e:
    print(f"Got expected error: {e}")